# Tiled polygonization tutorial

Polygonize a raster in tiles with `contourrs.shapes_arrow`, merge
same-class neighbors, and visualize the result.

This notebook uses a small synthetic categorical raster so it runs
anywhere without downloading external data. An optional final cell
shows how to fetch real USDA CDL data.

In [ ]:
import tempfile
from pathlib import Path

import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from contourrs import shapes_arrow
from rasterio.transform import from_origin
from rasterio.windows import Window

matplotlib.use("Agg")

## 1) Create a synthetic categorical raster

A 256x256 raster with 7 random land-cover classes, saved as a GeoTIFF.

In [ ]:
rng = np.random.default_rng(42)
synthetic = rng.integers(1, 8, size=(256, 256), dtype=np.uint8)
transform = from_origin(500000.0, 4500000.0, 30.0, 30.0)

tmp_dir = Path(tempfile.mkdtemp(prefix="contourrs-cdl-"))
raster_path = tmp_dir / "synthetic_cdl.tif"

with rasterio.open(
    raster_path,
    "w",
    driver="GTiff",
    height=synthetic.shape[0],
    width=synthetic.shape[1],
    count=1,
    dtype=synthetic.dtype,
    crs="EPSG:32615",
    transform=transform,
    nodata=0,
) as dst:
    dst.write(synthetic, 1)

print(f"Wrote synthetic raster: {raster_path}")
print(f"Shape: {synthetic.shape}, classes: {np.unique(synthetic).tolist()}")

## 2) Helper: transform tuple

`contourrs` expects a 6-element affine transform tuple `(a, b, c, d, e, f)`.
Rasterio's `Affine` object needs a quick conversion.

In [ ]:
def transform_tuple(t):
    return (t.a, t.b, t.c, t.d, t.e, t.f)

## 3) Polygonize in tiles

Read the raster in 64x64 tiles, polygonize each tile with
`shapes_arrow`, and collect the results into one GeoDataFrame.

In [ ]:
tile_size = 64
connectivity = 4
frames = []

with rasterio.open(raster_path) as src:
    rows = range(0, src.height, tile_size)
    cols = range(0, src.width, tile_size)
    total_tiles = len(rows) * len(cols)
    print(
        f"Raster {src.width}x{src.height}, tile_size={tile_size}, tiles={total_tiles}"
    )

    for row_off in rows:
        for col_off in cols:
            window = Window.from_slices(
                (row_off, min(row_off + tile_size, src.height)),
                (col_off, min(col_off + tile_size, src.width)),
            )
            block = src.read(1, window=window)
            mask = block != 0
            if src.nodata is not None:
                mask &= block != src.nodata
            if not mask.any():
                continue

            table = shapes_arrow(
                block,
                mask=mask,
                connectivity=connectivity,
                transform=transform_tuple(src.window_transform(window)),
            )
            if table.num_rows == 0:
                continue

            tile_gdf = gpd.GeoDataFrame.from_arrow(table)
            tile_gdf = gpd.GeoDataFrame(
                {"value": tile_gdf["value"], "geometry": tile_gdf.geometry},
                geometry="geometry",
                crs=tile_gdf.crs,
            )
            frames.append(tile_gdf)

    crs = src.crs

tiled = gpd.GeoDataFrame(
    pd.concat(frames, ignore_index=True),
    geometry="geometry",
)
tiled["value"] = tiled["value"].astype("int32")
tiled.set_crs(crs, inplace=True)

print(f"Raw tiled polygons: {len(tiled):,}")

## 4) Merge same-class neighbors

Dissolve polygons that share the same class value, then explode
multipolygons back to individual geometries. This merges
tile-boundary artifacts.

In [ ]:
merged = tiled.dissolve(by="value", as_index=False).explode(
    index_parts=False, ignore_index=True
)
merged = merged[~merged.geometry.is_empty]
merged["value"] = merged["value"].astype("int32")
merged = merged[["value", "geometry"]]

print(f"Merged polygons: {len(merged):,}")
print(f"Classes: {sorted(merged['value'].unique())}")

## 5) Plot: raster vs vector polygons

Side-by-side comparison of the original raster and the merged vector polygons.

In [ ]:
with rasterio.open(raster_path) as src:
    raster = src.read(1)
    bounds = src.bounds

extent = (bounds.left, bounds.right, bounds.bottom, bounds.top)
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(raster, cmap="tab20", interpolation="nearest", extent=extent)
axes[0].set_title("Categorical raster")

merged.plot(
    ax=axes[1],
    column="value",
    cmap="tab20",
    edgecolor="black",
    linewidth=0.03,
    legend=False,
)
axes[1].set_title(f"Merged polygons ({len(merged):,})")

for ax in axes:
    ax.set_axis_off()
    ax.set_xlim(bounds.left, bounds.right)
    ax.set_ylim(bounds.bottom, bounds.top)
    ax.set_aspect("equal")

plt.tight_layout()

## 6) Optional: real USDA CDL data

Disabled by default to keep CI deterministic.
Set `RUN_REAL = True` to fetch Cropland Data Layer for Polk County, IA
and run the same tiled polygonization pipeline.

In [ ]:
import re
from urllib.parse import urlencode
from urllib.request import urlopen, urlretrieve

RUN_REAL = False

if RUN_REAL:
    year, fips = 2023, "19153"
    base = "https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLFile"
    cdl_url = f"{base}?{urlencode({'year': year, 'fips': fips})}"

    with urlopen(cdl_url, timeout=60) as resp:  # noqa: S310
        payload = resp.read().decode("utf-8", errors="replace")
    match = re.search(r"<returnURL>([^<]+)</returnURL>", payload)
    assert match, "Could not parse CDL return URL"
    download_url = match.group(1).strip()

    real_raster = tmp_dir / f"cdl_{year}_{fips}.tif"
    if not real_raster.exists():
        print(f"Downloading: {download_url}")
        urlretrieve(download_url, real_raster)  # noqa: S310
    else:
        print(f"Using cached: {real_raster}")

    # Polygonize with the same tiled approach
    real_frames = []
    with rasterio.open(real_raster) as src:
        for row_off in range(0, src.height, 1024):
            for col_off in range(0, src.width, 1024):
                window = Window.from_slices(
                    (row_off, min(row_off + 1024, src.height)),
                    (col_off, min(col_off + 1024, src.width)),
                )
                block = src.read(1, window=window)
                mask = (block != 0) & (
                    block != src.nodata if src.nodata is not None else True
                )
                if not mask.any():
                    continue
                t = shapes_arrow(
                    block,
                    mask=mask,
                    connectivity=4,
                    transform=transform_tuple(src.window_transform(window)),
                )
                if t.num_rows > 0:
                    real_frames.append(gpd.GeoDataFrame.from_arrow(t))

    real_tiled = gpd.GeoDataFrame(
        pd.concat(real_frames, ignore_index=True), geometry="geometry"
    )
    real_merged = real_tiled.dissolve(by="value", as_index=False).explode(
        index_parts=False, ignore_index=True
    )
    print(f"Real CDL — raw: {len(real_tiled):,}, merged: {len(real_merged):,}")
else:
    print("Skipping real CDL download (set RUN_REAL = True to enable).")